In [1]:
import torch
from torch import nn

In [2]:
from torch.utils.tensorboard import SummaryWriter

In [3]:
writer = SummaryWriter()

In [4]:
import os
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Use all available CPU cores for data loading by default.
NUM_WORKERS = os.cpu_count() or 0

def create_dataloader(train_dir: str,
                      test_dir: str,
                      transforms: transforms.Compose,
                      batch_size: int,
                      num_worksers: int = NUM_WORKERS):
    # Load training images from folder structure.
    train_data  = datasets.ImageFolder(root=train_dir,
                                       transform=transforms)

    # Load test images from folder structure.
    test_data = datasets.ImageFolder(root=test_dir,
                                     transform=transforms)

    # Get class names from training dataset folders.
    class_names = train_data.classes

    # Create DataLoader for training data (shuffled each epoch).
    train_dataloader = DataLoader(dataset=train_data,
                                  shuffle=True,
                                  batch_size=batch_size,
                                  num_workers=num_worksers)

    # Create DataLoader for test data (no shuffle for evaluation).
    test_dataloader =  DataLoader(dataset=test_data,
                                  batch_size=batch_size,
                                  shuffle=False,
                                  num_workers=num_worksers)

    # Return both dataloaders and class labels.
    return train_dataloader, test_dataloader, class_names


In [5]:
class DesertClassifier(nn.Module):
    def __init__(self, input_shape:int, hidden_unit:int, output_shape:int):
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape, out_channels=hidden_unit, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(num_features=hidden_unit),
            nn.GELU(),
            nn.Conv2d(in_channels=hidden_unit, out_channels=hidden_unit, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(num_features=hidden_unit),
            nn.GELU(),
            nn.MaxPool2d(kernel_size=2,stride=2)
        )
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_unit, out_channels=hidden_unit, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(num_features=hidden_unit),
            nn.GELU(),
            nn.Conv2d(in_channels=hidden_unit, out_channels=hidden_unit, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(num_features=hidden_unit),
            nn.GELU(),
            nn.MaxPool2d(kernel_size=2,stride=2)
        )
        self.dense_layer = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.5),
            nn.Linear(in_features=hidden_unit *16*16, out_features=output_shape)
        )
    def forward(self,x):
        out = self.conv_block_1(x)
        out = self.conv_block_2(out)
        out = self.dense_layer(out)
        return out

In [6]:
def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
              ):
    model.train() # Modelimizi train moduna alıyoruz.
    
    train_loss = 0 # Train Loss değerlerini tutmak için bir değişken oluşturuyoruz.
    train_acc = 0 # Train Accuracy değerlerini tutmak için bir değişken oluşturuyoruz.

    for batch, (X,y) in enumerate(dataloader): # Batch size gerekli değil burada.
        y_pred = model(X) # Modelimize bir tahminde bulunduruyoruz.
        
        loss = loss_fn(y_pred,y) # Loss değerlerimizi loss_fn ile hesaplıyoruz.
        train_loss += loss.item() # Çıkan loss değerlerini train_loss değişlenine toplayarak atıyoruz.

        # Modelimizi backpropagation yapıyoruz.
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Softmax kullanarak modelimize label tahmininde bulunduruyoruz.
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1) 
        
        train_acc += (y_pred_class == y).sum().item() / len(y_pred) # Accuracy değerlerimizi bir değişkende tutuyoruz.

    train_loss /= len(dataloader) # Train Loss değerlerimizi dataloader boyuna bölüyoruz ve ort. elde ediyoruz.
    train_acc /= len(dataloader) # Train Acc değerlerimizi dataloader boyuna bölüyoruz ve ort. elde ediyoruz
    return train_loss, train_acc # Geriye train_loss ve train_acc döndürüyoruz.

def test_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
              ):
    model.eval() # Modelimizi test moduna alıyoruz.
    
    test_loss = 0 # test loss'ları tutmak için bir değişken oluşturuyoruz.
    test_acc = 0 # test accuracy'ları tutmak için bir değişken oluşturuyoruz.
    
    with torch.inference_mode(): # inference mode'a aldık.
        for batch, (X,y) in enumerate(dataloader): # batch gerekli değil fakat yine de aldık.
            test_pred = model(X) # modelimize tahmin ettiriyoruz.
            
            loss = loss_fn(test_pred,y) # loss'umuzu loss_fn ile hesaplıyoruz.
            test_loss += loss.item() # loss değerlerimizi test_loss değişkeninde topluyoruz.

            # Softmax activation function ile label tahmin ettiriyoruz.
            test_pred_label = torch.softmax(test_pred,dim=1).argmax(dim=1) 
            
            acc = (test_pred_label == y).sum().item() / len(test_pred) # Calculate accuracy
            test_acc += acc # Accuracy değerlerimizi toplayıp test_acc değişkenine atıyoruz.
            
    test_loss /= len(dataloader) # Test loss değerlerimizi dataloader boyuna bölüyoruz.
    test_acc /= len(dataloader) # Test acc değerlerimizi dataloader boyuna bölüyoruz.
    
    return test_loss, test_acc # Geriye test_loss ve test_acc döndürüyoruz.

def train(model: torch.nn.Module,
               train_dataloader: torch.utils.data.DataLoader,
               test_dataloader: torch.utils.data.DataLoader,
               optimizer: torch.optim.Optimizer,
               loss_fn: torch.nn.Module = nn.CrossEntropyLoss(),
               epochs:int = 10,
               experiment_name:str = "experiment"
              ):
    results = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }
    for epoch in range(epochs):
        train_loss, train_acc = train_step(model = model,
                                           dataloader = train_dataloader,
                                           loss_fn = loss_fn,
                                           optimizer = optimizer,
                                          )
        test_loss, test_acc = test_step(model = model,
                                           dataloader = test_dataloader,
                                           loss_fn = loss_fn,
                                          )
        print(f""" 
        Epoch:{epoch}
        Train Loss : {train_loss:.2f} -  Train Accuracy : {train_acc*100:.2f}
        Test Loss  : { test_loss:.2f} -  Test Accuracy  : {test_acc*100:.2f}
        """)
        results["train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
        results["train_acc"].append(train_acc.item() if isinstance(train_acc, torch.Tensor) else train_acc)
        results["test_loss"].append(test_loss.item() if isinstance(test_loss, torch.Tensor) else test_loss)
        results["test_acc"].append(test_acc.item() if isinstance(test_acc, torch.Tensor) else test_acc)

        writer = SummaryWriter(log_dir=f"runs/{experiment_name}")
        writer.add_scalars(
            main_tag="Loss",
            tag_scalar_dict={
                "train_loss":train_loss,
                "test_loss": test_loss},
            global_step=epoch)

        writer.add_scalars(
            main_tag="Accuracy",
            tag_scalar_dict={
                "train_acc":train_acc,
                "test_acc": test_acc},
            global_step=epoch)
        
        writer.add_graph(model=model,input_to_model=torch.randn(32,3,64,64))

    writer.close()
        
    return results

In [7]:
NUM_EPOCHS = 5
BATCH_SIZE = 32
LEARNING_RATE = 0.0003

train_dir = "data/desert101/train"
test_dir = "data/desert101/test"

data_transform = transforms.Compose([
    transforms.Resize(size=(64,64)), 
    transforms.RandomHorizontalFlip(p=0.4),
    transforms.TrivialAugmentWide(),   
    transforms.ToTensor(),
    transforms.Normalize([0.5483, 0.4638, 0.3865],[0.2542, 0.2576, 0.2630]),
])

train_dataloader, test_dataloader, class_names= create_dataloader(
    train_dir=train_dir,
    test_dir=test_dir,
    batch_size=BATCH_SIZE,
    transforms=data_transform
)

model1 = DesertClassifier(
    input_shape=3,
    hidden_unit=10,
    output_shape=len(class_names)
)

model2 = DesertClassifier(
    input_shape=3,
    hidden_unit=32,
    output_shape=len(class_names)
)

In [8]:
loss_fn = torch.nn.CrossEntropyLoss()

In [9]:
train(
    model = model1,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    loss_fn=loss_fn,
    optimizer = torch.optim.Adam(params=model1.parameters(),lr=LEARNING_RATE),
    epochs=NUM_EPOCHS,
    experiment_name="hidden unit -> 10"
     )

 
        Epoch:0
        Train Loss : 1.52 -  Train Accuracy : 27.23
        Test Loss  : 1.39 -  Test Accuracy  : 18.19
        
 
        Epoch:1
        Train Loss : 1.43 -  Train Accuracy : 28.44
        Test Loss  : 1.36 -  Test Accuracy  : 40.95
        
 
        Epoch:2
        Train Loss : 1.35 -  Train Accuracy : 36.61
        Test Loss  : 1.36 -  Test Accuracy  : 32.69
        
 
        Epoch:3
        Train Loss : 1.39 -  Train Accuracy : 33.12
        Test Loss  : 1.29 -  Test Accuracy  : 38.22
        
 
        Epoch:4
        Train Loss : 1.34 -  Train Accuracy : 34.73
        Test Loss  : 1.32 -  Test Accuracy  : 40.46
        


{'train_loss': [1.5247533559799193,
  1.4336670637130737,
  1.3505231738090515,
  1.3867596626281737,
  1.3397854685783386],
 'train_acc': [0.27232142857142855,
  0.284375,
  0.36607142857142855,
  0.33125,
  0.34732142857142856],
 'test_loss': [1.3864136536916096,
  1.361364205678304,
  1.3635727564493816,
  1.2913703521092732,
  1.3201241493225098],
 'test_acc': [0.18189102564102563,
  0.4094551282051282,
  0.3269230769230769,
  0.3822115384615385,
  0.4046474358974359]}

In [10]:
train(
    model = model2,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    loss_fn=loss_fn,
    optimizer = torch.optim.Adam(params=model2.parameters(),lr=LEARNING_RATE),
    epochs=NUM_EPOCHS,
    experiment_name="hidden unit -> 32"
     )

 
        Epoch:0
        Train Loss : 1.61 -  Train Accuracy : 31.96
        Test Loss  : 1.38 -  Test Accuracy  : 22.92
        
 
        Epoch:1
        Train Loss : 1.45 -  Train Accuracy : 34.82
        Test Loss  : 1.45 -  Test Accuracy  : 28.12
        
 
        Epoch:2
        Train Loss : 1.40 -  Train Accuracy : 36.34
        Test Loss  : 1.42 -  Test Accuracy  : 27.08
        
 
        Epoch:3
        Train Loss : 1.28 -  Train Accuracy : 43.21
        Test Loss  : 1.40 -  Test Accuracy  : 31.17
        
 
        Epoch:4
        Train Loss : 1.21 -  Train Accuracy : 46.47
        Test Loss  : 1.39 -  Test Accuracy  : 42.55
        


{'train_loss': [1.6096871018409729,
  1.453761065006256,
  1.4044281125068665,
  1.277739930152893,
  1.2149819374084472],
 'train_acc': [0.3196428571428572,
  0.3482142857142857,
  0.3633928571428572,
  0.4321428571428571,
  0.46473214285714287],
 'test_loss': [1.3812641302744548,
  1.4474093516667683,
  1.416533390680949,
  1.397836168607076,
  1.3875893354415894],
 'test_acc': [0.22916666666666666,
  0.28125,
  0.2708333333333333,
  0.31169871794871795,
  0.4254807692307692]}

In [27]:
import os

# Tensorboard'un yerini manuel olarak sisteme tanıtıyoruz
# Bu komut 'tensorboard' dosyasını aramayı bırak, doğrudan python üzerinden çalıştır der.
os.environ['TENSORBOARD_BINARY'] = '/home/abdulkadir/.local/bin/tensorboard'

In [28]:
import torch.utils.tensorboard
%reload_ext tensorboard
%tensorboard --logdir runs